In [1]:
import torchvision
import torchvision.transforms.v2 as T

In [2]:
import torch

By default, the FashionMNIST class loads
images as PIL (Python Image Library) images, with integer pixel values
ranging from 0 to 255. But we need PyTorch float tensors instead, with
scaled pixel values. Luckily, TorchVision datasets accept a transform
argument which lets you pass a preprocessing function that will get
executed on the fly whenever the data is accessed (there’s also a
target_transform argument if you need to preprocess the targets).
TorchVision provides many transform objects that you can use for this
(most of these transforms are PyTorch modules).
In this code, we create a Compose transform to chain two transforms: a
ToImage transform followed by a ToDtype transform. ToImage converts
various formats—including PIL images, NumPy arrays, and tensors—to
TorchVision’s Image class, which is a subclass of Tensor. The ToDtype
transform converts the data type, in this case to 32-bit floats. We also set its
scale argument to True to ensure the values get scaled between 0.0 and
1.0.⁠
1

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


In [4]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale = True)])
train_and_valid_data = torchvision.datasets.FashionMNIST(
root="datasets", train=True, download=True, transform=toTensor)

test_data = torchvision.datasets.FashionMNIST(
root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])


In [5]:
from torch.utils.data import DataLoader
import torch.nn as nn
torch.manual_seed(42)

In [6]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)
valid_loader = DataLoader(valid_data, batch_size=32)


In [7]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )
    def forward(self, X):
        return self.mlp(X)

In [8]:
model = ImageClassifier(n_inputs=28 * 28, n_hidden1=300, 
                        n_hidden2=100, 
                        n_classes=10)
xentropy = nn.CrossEntropyLoss()

In [12]:
model = model.to(device)  

In [13]:
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [14]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [20]:
train(model, optimizer, xentropy, train_loader, n_epochs=20)

Epoch 1/20, Loss: 0.2416
Epoch 2/20, Loss: 0.2388
Epoch 3/20, Loss: 0.2269
Epoch 4/20, Loss: 0.2228
Epoch 5/20, Loss: 0.2148
Epoch 6/20, Loss: 0.2081
Epoch 7/20, Loss: 0.2051
Epoch 8/20, Loss: 0.1962
Epoch 9/20, Loss: 0.1915
Epoch 10/20, Loss: 0.1885
Epoch 11/20, Loss: 0.1806
Epoch 12/20, Loss: 0.1784
Epoch 13/20, Loss: 0.1720
Epoch 14/20, Loss: 0.1671
Epoch 15/20, Loss: 0.1663
Epoch 16/20, Loss: 0.1618
Epoch 17/20, Loss: 0.1563
Epoch 18/20, Loss: 0.1500
Epoch 19/20, Loss: 0.1488
Epoch 20/20, Loss: 0.1465


In [16]:
import torchmetrics

In [17]:
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

In [18]:
def evaluate(model, data_loader, metric):
    model.eval()
    metric.reset()          # clear any previous state
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)   # update with batch predictions
    return metric.compute()                  # final accuracy

In [21]:
val_acc = evaluate(model, valid_loader, accuracy)
print(f"Validation accuracy: {val_acc:.4f}")

test_acc = evaluate(model, test_loader, accuracy)
print(f"Test accuracy: {test_acc:.4f}")

Validation accuracy: 0.8872
Test accuracy: 0.8933
